# Cranfield Semantic and Hybrid Retrieval

## Objective
This notebook demonstrates semantic document retrieval using a
Sentence Transformer and hybrid ranking using Reciprocal Rank Fusion (RRF).

We will:
1. Load the Cranfield dataset.
2. Generate semantic document embeddings.
3. Retrieve documents for a sample query.
4. Combine TF-IDF, BM25, and semantic rankings using RRF.
5. Inspect the final ranked results.

In [5]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root.resolve()))

from src.data_loader import load_cranfield

documents, queries, qrels = load_cranfield()

print(f"Documents: {len(documents)}")
print(f"Queries: {len(queries)}")
print(f"Relevance judgments: {len(qrels)}")

documents.head()

Documents: 1400
Queries: 225
Relevance judgments: 1837


,doc_id,title,author,bib,text
0,1,experimental investigation of the aerodynamics...,"brenckman,m.","j. ae. scs. 25, 1958, 324.",experimental investigation of the aerodynamics...
1,2,simple shear flow past a flat plate in an inco...,ting-yili,"department of aeronautical engineering, rensse...",simple shear flow past a flat plate in an inco...
2,3,the boundary layer in simple shear flow past a...,m. b. glauert,"department of mathematics, university of manch...",the boundary layer in simple shear flow past a...
3,4,approximate solutions of the incompressible la...,"yen,k.t.","j. ae. scs. 22, 1955, 728.",approximate solutions of the incompressible la...
4,5,one-dimensional transient heat conduction into...,"wasserman,b.","j. ae. scs. 24, 1957, 924.",one-dimensional transient heat conduction into...


## 2. Semantic Retrieval with Sentence Transformers

We use the pre-trained `all-MiniLM-L6-v2` model to represent documents
and queries as dense vector embeddings. Cosine similarity ranks documents
according to semantic similarity to the query.

Document embeddings are generated once when the retriever is initialized,
so they can be reused across multiple searches.

In [6]:
from src.semantic_search import SemanticRetriever

semantic_retriever = SemanticRetriever(documents)

print(f"Documents indexed semantically: {len(semantic_retriever.doc_ids)}")
print(f"Embedding matrix shape: {semantic_retriever.document_embeddings.shape}")

c:\Users\Dell\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 44/44 [01:34<00:00,  2.14s/it]

Documents indexed semantically: 1400
Embedding matrix shape: (1400, 384)


In [7]:
query = "aircraft wing design and airflow"

semantic_results = semantic_retriever.semantic_search(
    query,
    top_k=5,
)

print(f"Query: {query}\n")
print("Top 5 semantic search results:\n")

for rank, (doc_id, score) in enumerate(semantic_results, start=1):
    document = documents.loc[documents["doc_id"].astype(str) == doc_id].iloc[0]

    print(f"{rank}. Document ID: {doc_id}")
    print(f"   Similarity score: {score:.4f}")
    print(f"   Title: {document['title']}")
    print()

Query: aircraft wing design and airflow

Top 5 semantic search results:

1. Document ID: 675
   Similarity score: 0.6561
   Title: pressure distribution and surface flow on 5( and 9( thick wings with
curved tip and 60degree sweepback .

2. Document ID: 222
   Similarity score: 0.6491
   Title: the flow over delta wings at low speeds with leading
edge separation .

3. Document ID: 601
   Similarity score: 0.6270
   Title: calculation of the flow past slender delta wings with
leading edge separation .

4. Document ID: 612
   Similarity score: 0.6213
   Title: pressure distributions and flow patterns at m=4 . on
some delta wings of inverted 'v' cross section .

5. Document ID: 794
   Similarity score: 0.6197
   Title: experiments with a tapered swept-back wing of warren 12 planform at mach
 numbers between 0.6 and 1.6 .



In [8]:
import inspect
from src.tfidf_bm25 import LexicalRetriever

print(inspect.signature(LexicalRetriever))

(documents, preprocessor: src.preprocessing.TextPreprocessor, index_api: src.search_api.IndexAPI, k1: float = 1.5, b: float = 0.75, id_column: str = None, text_column: str = None)


In [9]:
import inspect
from src.preprocessing import TextPreprocessor
from src.search_api import IndexAPI

print("TextPreprocessor:", inspect.signature(TextPreprocessor))
print("IndexAPI:", inspect.signature(IndexAPI))

TextPreprocessor: ()
IndexAPI: (index)


In [10]:
import inspect
from src.indexing import InvertedIndex

print("InvertedIndex constructor:", inspect.signature(InvertedIndex))
print("\nIndexing module functions:")
print([
    name for name, obj in inspect.getmembers(__import__("src.indexing", fromlist=["*"]))
    if inspect.isfunction(obj) and not name.startswith("_")
])

InvertedIndex constructor: ()

Indexing module functions:
[]


In [11]:
import inspect
from src.indexing import InvertedIndex

print("InvertedIndex methods:")
print([
    name for name, obj in inspect.getmembers(InvertedIndex, predicate=inspect.isfunction)
    if not name.startswith("_")
])

InvertedIndex methods:
['add_document', 'build', 'get_all_documents', 'get_document_frequency', 'get_document_length', 'get_documents', 'get_term_frequency', 'get_vocabulary', 'load', 'save']


In [12]:
import inspect
from src.indexing import InvertedIndex

print(inspect.signature(InvertedIndex.build))
print(inspect.getsource(InvertedIndex.build))

(self, documents, preprocessor)
    def build(self, documents, preprocessor):
        """
        Build the inverted index from a collection of documents.
        """

        for _, document in documents.iterrows():

            # Combine title and abstract
            text = (
                document["title"]
                + " "
                + document["text"]
            )

            tokens = preprocessor.preprocess(text)

            self.add_document(
                document["doc_id"],
                tokens
            )



In [13]:
from src.preprocessing import TextPreprocessor
from src.indexing import InvertedIndex
from src.search_api import IndexAPI
from src.tfidf_bm25 import LexicalRetriever

# Preprocess and index the Cranfield documents
preprocessor = TextPreprocessor()
inverted_index = InvertedIndex()
inverted_index.build(documents, preprocessor)

# Create the search API and lexical retriever
index_api = IndexAPI(inverted_index)

lexical_retriever = LexicalRetriever(
    documents=documents,
    preprocessor=preprocessor,
    index_api=index_api,
)

print("Lexical retriever initialized successfully.")

Lexical retriever initialized successfully.


In [14]:
from src.hybrid_search import HybridRetriever

# Combine lexical and semantic retrieval
hybrid_retriever = HybridRetriever(
    lexical_retriever=lexical_retriever,
    semantic_retriever=semantic_retriever,
)

query = "aircraft wing design and airflow"

hybrid_results = hybrid_retriever.search(
    query,
    top_k=5,
    candidate_k=100,
)

print(f"Query: {query}")
print("\nTop 5 hybrid RRF results:\n")

for rank, (doc_id, score) in enumerate(hybrid_results, start=1):
    document = documents.loc[
        documents["doc_id"].astype(str) == doc_id
    ].iloc[0]

    print(f"{rank}. Document ID: {doc_id}")
    print(f"   RRF score: {score:.4f}")
    print(f"   Title: {document['title']}")
    print()

Query: aircraft wing design and airflow

Top 5 hybrid RRF results:

1. Document ID: 415
   RRF score: 0.0427
   Title: the aerodynamic design of section shapes for swept wings .

2. Document ID: 612
   RRF score: 0.0408
   Title: pressure distributions and flow patterns at m=4 . on
some delta wings of inverted 'v' cross section .

3. Document ID: 792
   RRF score: 0.0408
   Title: some low speed problems of high speed aircraft .

4. Document ID: 924
   RRF score: 0.0375
   Title: a method for calculating the lift and centre of pressure
of wing-body-tail combinations at subsonic, transonic
speeds .

5. Document ID: 1144
   RRF score: 0.0373
   Title: slipstream flow around several tilt-wing vtol aircraft models
operating near the ground .



## 4. Semantic vs. Hybrid Retrieval

Semantic retrieval ranks documents using embedding similarity.
Hybrid retrieval combines the TF-IDF, BM25, and semantic rankings
using Reciprocal Rank Fusion (RRF).

The rankings may differ because RRF combines the relative positions
of documents across retrieval systems rather than directly adding
their original similarity scores.

In [16]:
import pandas as pd

semantic_comparison = pd.DataFrame(
    [
        {
            "doc_id": doc_id,
            "semantic_rank": rank,
            "semantic_score": round(score, 4),
        }
        for rank, (doc_id, score) in enumerate(semantic_results, start=1)
    ]
)

hybrid_comparison = pd.DataFrame(
    [
        {
            "doc_id": doc_id,
            "hybrid_rank": rank,
            "rrf_score": round(score, 4),
        }
        for rank, (doc_id, score) in enumerate(hybrid_results, start=1)
    ]
)

comparison = semantic_comparison.merge(
    hybrid_comparison,
    on="doc_id",
    how="outer",
)

comparison = comparison.sort_values(
    by=["hybrid_rank", "semantic_rank"],
    na_position="last",
)

comparison = comparison.merge(
    documents[["doc_id", "title"]].assign(
        doc_id=lambda df: df["doc_id"].astype(str)
    ),
    on="doc_id",
    how="left",
)

comparison[
    [
        "doc_id",
        "title",
        "semantic_rank",
        "semantic_score",
        "hybrid_rank",
        "rrf_score",
    ]
]

,doc_id,title,semantic_rank,semantic_score,hybrid_rank,rrf_score
0,415,the aerodynamic design of section shapes for s...,NaN,NaN,1.0,0.0427
1,612,pressure distributions and flow patterns at m=...,4.0,0.6213,2.0,0.0408
2,792,some low speed problems of high speed aircraft .,NaN,NaN,3.0,0.0408
3,924,a method for calculating the lift and centre o...,NaN,NaN,4.0,0.0375
4,1144,slipstream flow around several tilt-wing vtol ...,NaN,NaN,5.0,0.0373
5,675,pressure distribution and surface flow on 5( a...,1.0,0.6561,NaN,NaN
6,222,the flow over delta wings at low speeds with l...,2.0,0.6491,NaN,NaN
7,601,calculation of the flow past slender delta win...,3.0,0.6270,NaN,NaN
8,794,experiments with a tapered swept-back wing of ...,5.0,0.6197,NaN,NaN
